# fact_resumen_reporte_calidad — Métricas de Calidad por Ámbito

Este notebook consulta la tabla de resumen de calidad y presenta los resultados
organizados por cada nivel de análisis (ámbito) disponible en el modelo.

## ¿Qué hace este notebook?

Lee la tabla `uc_axa_cli.gold.fact_resumen_reporte_calidad` y presenta las métricas
de calidad desde diferentes perspectivas: cuántos registros se evaluaron, cuántos
pasaron las reglas, cuántos fallaron y cuál es el porcentaje de calidad.

## Tabla fuente

`uc_axa_cli.gold.fact_resumen_reporte_calidad`

## Columnas principales

| Columna | Qué representa |
|---|---|
| `ambito` | Nivel de agrupación del resultado (fuente, cliente, dimension_regla, atributo, pk_regla_calidad) |
| `descripcion_ambito` | Nombre del elemento evaluado (ej: nombre del satélite, nombre de la regla) |
| `cant_evaluados` | Total de registros que pasaron por las reglas de calidad |
| `cant_validos` | Registros que cumplieron todas las reglas |
| `cant_invalidos` | Registros que fallaron al menos una regla |
| `cant_remediados` | Registros inválidos que ya fueron corregidos |

## Cálculo del porcentaje de calidad

```
porcentaje_calidad = cant_validos / cant_evaluados * 100
```

## Semáforo de calidad

| Color | Rango | Interpretación |
|---|---|---|
| 🟢 VERDE | ≥ 95% | Calidad aceptable |
| 🟡 AMARILLO | ≥ 80% y < 95% | Requiere atención |
| 🔴 ROJO | < 80% | Calidad crítica — acción inmediata |

## Paso 1 — Diagnóstico: ¿qué ámbitos existen en la tabla?

Antes de consultar por ámbito, verificamos los valores exactos que tiene
la columna `ambito` en la tabla. Esto es importante porque los nombres
deben coincidir exactamente con los filtros de las consultas siguientes.

In [ ]:
%sql
SELECT DISTINCT ambito, COUNT(*) AS filas
FROM uc_axa_cli.gold.fact_resumen_reporte_calidad
GROUP BY ambito
ORDER BY ambito

## Paso 2 — Resumen ejecutivo: una fila por ámbito

Esta consulta da una **vista de alto nivel** del estado de calidad.
Muestra una sola fila por cada ámbito con los totales consolidados.

Es útil para responder preguntas como:
- ¿Cuál ámbito tiene el mayor volumen de registros evaluados?
- ¿Qué ámbito tiene el porcentaje de calidad más bajo?
- ¿Cuántos registros en total han sido remediados?

Los ámbitos aparecen ordenados de mayor a menor impacto en el negocio:
primero fuente, luego cliente, dimension_regla, atributo y regla.

In [ ]:
%sql
SELECT
    ambito,
    COUNT(DISTINCT descripcion_ambito)         AS total_elementos,
    SUM(cant_evaluados)                        AS total_evaluados,
    SUM(cant_validos)                          AS total_validos,
    SUM(cant_invalidos)                        AS total_invalidos,
    SUM(cant_remediados)                       AS total_remediados,
    ROUND(SUM(cant_validos) * 100.0 / NULLIF(SUM(cant_evaluados), 0), 2) AS porcentaje_calidad
FROM  uc_axa_cli.gold.fact_resumen_reporte_calidad
GROUP BY ambito
ORDER BY
    CASE ambito
        WHEN 'fuente'           THEN 1
        WHEN 'cliente'          THEN 2
        WHEN 'dimension_regla'  THEN 3
        WHEN 'atributo'         THEN 4
        WHEN 'pk_regla_calidad' THEN 5
        ELSE 6
    END

## Paso 3 — Detalle por FUENTE (satélite)

Muestra la calidad agrupada por **fuente de datos**, es decir, por cada satélite
o tabla que fue evaluada por las reglas de calidad.

Permite identificar qué fuente de datos tiene los mayores problemas de calidad.
Los resultados aparecen ordenados de menor a mayor porcentaje para ver primero
los más críticos.

In [ ]:
%sql
SELECT
    'fuente'            AS ambito,
    descripcion_ambito  AS fuente,
    SUM(cant_evaluados) AS evaluados,
    SUM(cant_validos)   AS validos,
    SUM(cant_invalidos) AS invalidos,
    SUM(cant_remediados)AS remediados,
    ROUND(SUM(cant_validos) * 100.0 / NULLIF(SUM(cant_evaluados), 0), 2) AS porcentaje_calidad
FROM  uc_axa_cli.gold.fact_resumen_reporte_calidad
WHERE ambito = 'fuente'
GROUP BY descripcion_ambito
ORDER BY porcentaje_calidad ASC

## Paso 4 — Detalle por CLIENTE

Muestra la calidad agrupada por **cliente**, permitiendo identificar qué clientes
específicos tienen datos con problemas de calidad.

Este ámbito es el más granular a nivel de negocio: permite responder
"¿cuáles clientes tienen datos incompletos o inválidos?"

In [ ]:
%sql
SELECT
    'cliente'           AS ambito,
    descripcion_ambito  AS cliente,
    SUM(cant_evaluados) AS evaluados,
    SUM(cant_validos)   AS validos,
    SUM(cant_invalidos) AS invalidos,
    SUM(cant_remediados)AS remediados,
    ROUND(SUM(cant_validos) * 100.0 / NULLIF(SUM(cant_evaluados), 0), 2) AS porcentaje_calidad
FROM  uc_axa_cli.gold.fact_resumen_reporte_calidad
WHERE ambito = 'cliente'
GROUP BY descripcion_ambito
ORDER BY porcentaje_calidad ASC

## Paso 5 — Detalle por DIMENSIÓN DE REGLA

Muestra la calidad agrupada por **dimensión o categoría de regla**: completitud,
formato, unicidad, consistencia, etc.

Permite entender **qué tipo de problema de calidad** es el más frecuente.
Por ejemplo: ¿es un problema de campos vacíos (completitud) o de formatos
incorrectos (formato)?

In [ ]:
%sql
SELECT
    'dimension_regla'   AS ambito,
    descripcion_ambito  AS dimension_regla,
    SUM(cant_evaluados) AS evaluados,
    SUM(cant_validos)   AS validos,
    SUM(cant_invalidos) AS invalidos,
    SUM(cant_remediados)AS remediados,
    ROUND(SUM(cant_validos) * 100.0 / NULLIF(SUM(cant_evaluados), 0), 2) AS porcentaje_calidad
FROM  uc_axa_cli.gold.fact_resumen_reporte_calidad
WHERE ambito = 'dimension_regla'
GROUP BY descripcion_ambito
ORDER BY porcentaje_calidad ASC

## Paso 6 — Detalle por ATRIBUTO

Muestra la calidad agrupada por **campo o columna** que fue evaluado.

Permite identificar exactamente **qué campo tiene más problemas**: por ejemplo,
si el campo `numero_documento` tiene un porcentaje bajo, significa que muchos
registros tienen ese dato vacío o mal formado.

In [ ]:
%sql
SELECT
    'atributo'          AS ambito,
    descripcion_ambito  AS atributo,
    SUM(cant_evaluados) AS evaluados,
    SUM(cant_validos)   AS validos,
    SUM(cant_invalidos) AS invalidos,
    SUM(cant_remediados)AS remediados,
    ROUND(SUM(cant_validos) * 100.0 / NULLIF(SUM(cant_evaluados), 0), 2) AS porcentaje_calidad
FROM  uc_axa_cli.gold.fact_resumen_reporte_calidad
WHERE ambito = 'atributo'
GROUP BY descripcion_ambito
ORDER BY porcentaje_calidad ASC

## Paso 7 — Detalle por REGLA DE CALIDAD

Muestra la calidad agrupada por **regla individual** que fue evaluada.

Es el nivel más detallado: permite ver exactamente cuál regla específica
está fallando más. Por ejemplo: "el campo email no puede estar vacío"
o "el tipo de documento debe ser CC, CE o PA".

In [ ]:
%sql
SELECT
    'pk_regla_calidad'  AS ambito,
    descripcion_ambito  AS regla,
    SUM(cant_evaluados) AS evaluados,
    SUM(cant_validos)   AS validos,
    SUM(cant_invalidos) AS invalidos,
    SUM(cant_remediados)AS remediados,
    ROUND(SUM(cant_validos) * 100.0 / NULLIF(SUM(cant_evaluados), 0), 2) AS porcentaje_calidad
FROM  uc_axa_cli.gold.fact_resumen_reporte_calidad
WHERE ambito = 'pk_regla_calidad'
GROUP BY descripcion_ambito
ORDER BY porcentaje_calidad ASC

## Paso 8 — Vista completa con semáforo de calidad

Esta consulta presenta **todos los ámbitos juntos** en un solo resultado,
con una columna adicional `semaforo_calidad` que clasifica visualmente
el estado de cada elemento:

- 🟢 **VERDE**: calidad igual o superior al 95% — dentro del umbral aceptable
- 🟡 **AMARILLO**: calidad entre 80% y 94% — requiere seguimiento
- 🔴 **ROJO**: calidad por debajo del 80% — acción correctiva inmediata

Esta vista es ideal para exportar a Excel o alimentar un dashboard de calidad.

In [ ]:
%sql
SELECT
    ambito,
    descripcion_ambito,
    cant_evaluados,
    cant_validos,
    cant_invalidos,
    cant_remediados,
    ROUND(cant_validos * 100.0 / NULLIF(cant_evaluados, 0), 2) AS porcentaje_calidad,
    CASE
        WHEN ROUND(cant_validos * 100.0 / NULLIF(cant_evaluados, 0), 2) >= 95 THEN 'VERDE'
        WHEN ROUND(cant_validos * 100.0 / NULLIF(cant_evaluados, 0), 2) >= 80 THEN 'AMARILLO'
        ELSE 'ROJO'
    END AS semaforo_calidad
FROM  uc_axa_cli.gold.fact_resumen_reporte_calidad
ORDER BY
    CASE ambito
        WHEN 'fuente'           THEN 1
        WHEN 'cliente'          THEN 2
        WHEN 'dimension_regla'  THEN 3
        WHEN 'atributo'         THEN 4
        WHEN 'pk_regla_calidad' THEN 5
        ELSE 6
    END,
    porcentaje_calidad ASC